In [1]:
import spacy
from spacy.tokens import Span

In [2]:
nlp = spacy.load("en_core_web_sm")

In [3]:
text = """
Python and AWS are used for data analysis.
Microsoft works with Google Cloud.
PostgreSQL is used as a database.
New York is a location.
"""

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "→", ent.label_)

AWS → ORG
Microsoft → ORG
Google Cloud → PRODUCT
PostgreSQL → GPE
New York → GPE


# create custom entity

In [4]:
def add_custom_entities(doc):

    entities = []

    for token in doc:

        text = token.text.lower()

        if text == "python":
            entities.append(
                Span(doc, token.i, token.i + 1, label="SKILL")
            )

        elif text == "aws":
            entities.append(
                Span(doc, token.i, token.i + 1, label="CLOUD_PLATFORM")
            )

        elif text == "postgresql":
            entities.append(
                Span(doc, token.i, token.i + 1, label="DATABASE")
            )

    doc.ents = entities

    return doc

In [5]:
from spacy.matcher import PhraseMatcher
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

In [6]:
skill_patterns = [
    "Python",
    "Java",
    "SQL",
    "JavaScript"
]

tool_patterns = [
    "Power BI",
    "Tableau",
    "Excel"
]

database_patterns = [
    "PostgreSQL",
    "MySQL",
    "MongoDB",
    "Oracle"
]

cloud_patterns = [
    "AWS",
    "Azure",
    "GCP",
    "Google Cloud"
]

In [7]:
matcher.add(
    "SKILL",
    [nlp.make_doc(x) for x in skill_patterns]
)

matcher.add(
    "TOOL",
    [nlp.make_doc(x) for x in tool_patterns]
)

matcher.add(
    "DATABASE",
    [nlp.make_doc(x) for x in database_patterns]
)

matcher.add(
    "CLOUD_PLATFORM",
    [nlp.make_doc(x) for x in cloud_patterns]
)

In [8]:
text = """
We are looking for a Python developer with SQL experience.
The candidate should know Power BI and Excel.
Experience with PostgreSQL and AWS is required.
"""

doc = nlp(text)

matches = matcher(doc)

for match_id, start, end in matches:

    label = nlp.vocab.strings[match_id]
    entity = doc[start:end]

    print(entity.text, "→", label)

Python → SKILL
SQL → SKILL
Power BI → TOOL
Excel → TOOL
PostgreSQL → DATABASE
AWS → CLOUD_PLATFORM


In [9]:
entities = []

for match_id, start, end in matches:

    label = nlp.vocab.strings[match_id]

    span = Span(
        doc,
        start,
        end,
        label=label
    )

    entities.append(span)

doc.ents = entities
for ent in doc.ents:
    print(ent.text, "→", ent.label_)

Python → SKILL
SQL → SKILL
Power BI → TOOL
Excel → TOOL
PostgreSQL → DATABASE
AWS → CLOUD_PLATFORM


In [10]:
import pandas as pd 
df = pd.read_csv("clean_jobs_with_skills.csv")
text = df["clean_description"].iloc[0]

doc = nlp(text)

matches = matcher(doc)

for match_id, start, end in matches:

    label = nlp.vocab.strings[match_id]

    print(
        doc[start:end].text,
        "→",
        label
    )

In [11]:
taxonomy = pd.read_csv("Skill_taxonomy.csv")

print(taxonomy.head())
skills = (
    taxonomy["skill"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

    skill     category          aliases
0  Python  Programming   python3|python
1    Java  Programming             java
2       C  Programming                c
3     C++  Programming  cpp|c plus plus
4      C#  Programming   csharp|c sharp


# automatically create skill patterns

In [12]:
skill_patterns = [
    skill for skill in skills
    if skill
]
matcher.add(
    "SKILL",
    [nlp.make_doc(skill) for skill in skill_patterns]
)

In [13]:
text = """
We are looking for a Python developer with experience in AWS,
PostgreSQL and Power BI.
"""

doc = nlp(text)

matches = matcher(doc)

for match_id, start, end in matches:
    span = doc[start:end]
    print(span.text)

Python
AWS
AWS
PostgreSQL
PostgreSQL
Power BI
Power BI


In [14]:
for match_id, start, end in matches:
    span = doc[start:end]

    skill = span.text.lower()

    if skill == "aws":
        label = "CLOUD_PLATFORM"

    elif skill == "postgresql":
        label = "DATABASE"

    elif skill == "power bi":
        label = "TOOL"

    else:
        label = "SKILL"

    print(span.text, "→", label)

Python → SKILL
AWS → CLOUD_PLATFORM
AWS → CLOUD_PLATFORM
PostgreSQL → DATABASE
PostgreSQL → DATABASE
Power BI → TOOL
Power BI → TOOL


In [15]:
text = df.loc[1, "clean_description"]

doc = nlp(text)

matches = matcher(doc)

for match_id, start, end in matches:
    span = doc[start:end]
    print(span.text)  

basic computer knowledge
tamil
english
data entry
data processing
data collection


# Create save columns

In [16]:
def extract_ner_skills(text):
    doc = nlp(text)
    matches = matcher(doc)

    skills = []

    for match_id, start, end in matches:
        span = doc[start:end]

        skills.append(span.text)

    return list(set(skills))


In [17]:
df["ner_skills"] = df["clean_description"].apply(
    extract_ner_skills
)


In [18]:
df[["clean_description", "ner_skills"]].head()

,clean_description,ner_skills
0,job description send me jobs like this qualifi...,"[operations, bpo, data collection, kpo, englis..."
1,job description send me jobs like this qualifi...,"[data collection, english, tamil, data process..."
2,job description send me jobs like this as a de...,[sql]
3,job description send me jobs like this involve...,[c]
4,job description send me jobs like this please ...,"[git, java, angular, javascript, mysql, html]"


In [19]:
df.to_csv("clean_jobs_with_skills.csv", index=False)

# Day 13 Machine learning skill classifier

In [21]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [22]:
skills_df = pd.read_csv("skill_taxonomy.csv")
df = pd.read_csv("clean_jobs_with_skills.csv")
print(skills_df.head())
print(skills_df.columns)


    skill     category          aliases
0  Python  Programming   python3|python
1    Java  Programming             java
2       C  Programming                c
3     C++  Programming  cpp|c plus plus
4      C#  Programming   csharp|c sharp
Index(['skill', 'category', 'aliases'], dtype='str')


# Create label training data

In [58]:
def map_category(category):
    category = str(category).lower().strip()

    if category == "cloud":
        return "CLOUD"

    elif category == "database":
        return "DATABASE"

    elif category in ["data analytics", "office tools"]:
        return "BI_TOOL"

    else:
        return "SKILL"

In [24]:
skills_df["label"] = skills_df["category"].apply(map_category)

print(
    skills_df[["skill", "category", "label"]].head(20))


           skill         category     label
0         Python      Programming     SKILL
1           Java      Programming     SKILL
2              C      Programming     SKILL
3            C++      Programming     SKILL
4             C#      Programming     SKILL
5     JavaScript      Programming     SKILL
6     TypeScript      Programming     SKILL
7              R      Programming     SKILL
8           HTML  Web Development     SKILL
9            CSS  Web Development     SKILL
10         React  Web Development     SKILL
11       Angular  Web Development     SKILL
12        Vue.js  Web Development     SKILL
13       Node.js  Web Development     SKILL
14    Express.js  Web Development     SKILL
15       Next.js  Web Development     SKILL
16  Tailwind CSS  Web Development     SKILL
17           SQL         Database  DATABASE
18         MySQL         Database  DATABASE
19    PostgreSQL         Database  DATABASE


In [25]:
print(
    skills_df[
        skills_df["skill"].str.lower().isin(
            ["python", "aws", "postgresql", "power bi", "java", "mysql"]
        )
    ][["skill", "category", "label"]]
)
print(skills_df["label"].value_counts())

         skill        category     label
0       Python     Programming     SKILL
1         Java     Programming     SKILL
18       MySQL        Database  DATABASE
19  PostgreSQL        Database  DATABASE
23    Power BI  Data Analytics   BI_TOOL
43         AWS           Cloud     CLOUD
label
SKILL       80
BI_TOOL      8
CLOUD        7
DATABASE     6
Name: count, dtype: int64


In [26]:
skills_df["training_text"] = (
    skills_df["skill"].astype(str)
    + " "
    + skills_df["category"].astype(str)
)
print(
    skills_df[["training_text", "label"]].head(20)
)

                   training_text     label
0             Python Programming     SKILL
1               Java Programming     SKILL
2                  C Programming     SKILL
3                C++ Programming     SKILL
4                 C# Programming     SKILL
5         JavaScript Programming     SKILL
6         TypeScript Programming     SKILL
7                  R Programming     SKILL
8           HTML Web Development     SKILL
9            CSS Web Development     SKILL
10         React Web Development     SKILL
11       Angular Web Development     SKILL
12        Vue.js Web Development     SKILL
13       Node.js Web Development     SKILL
14    Express.js Web Development     SKILL
15       Next.js Web Development     SKILL
16  Tailwind CSS Web Development     SKILL
17                  SQL Database  DATABASE
18                MySQL Database  DATABASE
19           PostgreSQL Database  DATABASE


# Crete x and y

In [27]:
X = skills_df["skill"]
y = skills_df["label"]
print(y.value_counts())

label
SKILL       80
BI_TOOL      8
CLOUD        7
DATABASE     6
Name: count, dtype: int64


# training and testing data

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# create TF-IDF + Logistic Regression

In [29]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2)
    )),
    
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

# train nodel

In [30]:
model.fit(X_train, y_train)
print("Model trained successfully!")

Model trained successfully!


# make prdictions

In [31]:
y_pred = model.predict(X_test)

print(y_pred)

['SKILL' 'SKILL' 'SKILL' 'SKILL' 'SKILL' 'SKILL' 'SKILL' 'SKILL' 'SKILL'
 'SKILL' 'SKILL' 'SKILL' 'SKILL' 'SKILL' 'SKILL' 'SKILL' 'SKILL' 'CLOUD'
 'SKILL' 'SKILL' 'SKILL']


In [32]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.8571428571428571


In [33]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

     BI_TOOL       0.00      0.00      0.00         2
       CLOUD       1.00      1.00      1.00         1
    DATABASE       0.00      0.00      0.00         1
       SKILL       0.85      1.00      0.92        17

    accuracy                           0.86        21
   macro avg       0.46      0.50      0.48        21
weighted avg       0.74      0.86      0.79        21



c:\Users\HP\Documents\dataAnalyticstask\JobDescriptionproject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HP\Documents\dataAnalyticstask\JobDescriptionproject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HP\Documents\dataAnalyticstask\JobDescriptionproject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to

# create save column

In [34]:
def predict_ml_category(text):
    return model.predict([text])[0]
df["ml_category"] = df["clean_description"].apply(
    predict_ml_category
)

In [35]:
df[
    ["clean_description", "ml_category"]
].head()

,clean_description,ml_category
0,job description send me jobs like this qualifi...,SKILL
1,job description send me jobs like this qualifi...,SKILL
2,job description send me jobs like this as a de...,DATABASE
3,job description send me jobs like this involve...,SKILL
4,job description send me jobs like this please ...,SKILL


In [36]:
df.to_csv("clean_jobs_with_skills.csv", index=False)

In [37]:
print(df.columns.tolist())

['company', 'education', 'experience', 'industry', 'jobdescription', 'jobid', 'joblocation_address', 'jobtitle', 'numberofpositions', 'payrate', 'postdate', 'site_name', 'skills', 'uniq_id', 'clean_description', 'original_length', 'clean_length', 'dictionary_skills', 'regex_Phrase_skills', 'tfidf_keywords', 'ner_skills', 'ml_category']


# test again

In [38]:
test_phrases = [
    "Python programming",
    "AWS cloud platform",
    "PostgreSQL database",
    "Power BI dashboard",
    "Java programming",
    "MySQL database"
]

predictions = model.predict(test_phrases)

for text, prediction in zip(test_phrases, predictions):
    print(text, "→", prediction)

Python programming → SKILL
AWS cloud platform → CLOUD
PostgreSQL database → DATABASE
Power BI dashboard → BI_TOOL
Java programming → SKILL
MySQL database → DATABASE


In [39]:
joblib.dump(model, "skill_classifier.pkl")

print("skill_classifier.pkl saved successfully!")

skill_classifier.pkl saved successfully!


In [40]:
loaded_model = joblib.load("skill_classifier.pkl")

In [41]:
test = [
    "AWS cloud platform",
    "PostgreSQL database",
    "Python programming"
]

predictions = loaded_model.predict(test)

for text, prediction in zip(test, predictions):
    print(text, "→", prediction)

AWS cloud platform → CLOUD
PostgreSQL database → DATABASE
Python programming → SKILL
